# Example: STED-FM — Synaptic protein phenotypes

Compare a foundation model trained on STED microscopy with DINOv2 on super-resolution images of synaptic proteins.

**Datasets:** downloaded and reshaped by the helpers in [`sted_fm_utils/`](sted_fm_utils/):

- **NAS:** 224×224 crops of PSD-95 under four activity conditions (0MgGlyBic, 48hTTX, Block, GluGly).
- **Optim:** 224×224 crops of four proteins (CaMKII, PSD-95, actin, tubulin).
- **Zooniverse:** crops of individual synaptic protein clusters. This is the only set with masks suited to classical properties.

**Key steps:**
1. Install STED-FM and turn the chosen dataset into PhenoMe's folder layout.
2. Wrap the model. STED-FM expects one channel and does not use ImageNet normalization.
3. Extract embeddings, compute the `complete` property preset, and explore the projection.

To use your own images instead, skip the download and point `images_dir`, `masks_dir`, and `metadata_dir` at a folder with `images/`, optional `masks/`, and `metadata.csv`.


## 0. Install STED-FM

STED-FM is not a PhenoMe dependency. Install it in this environment before the cells below:

```bash
pip install "git+https://github.com/FLClab/STED-FM.git"
pip install requests
```

`requests` is used by `sted_fm_utils` to download the datasets. Details of the helper scripts are in [`sted_fm_utils/README.md`](sted_fm_utils/README.md).


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from phenome import PhenoMe, load_dinov2_model, make_dataframe_metadata_fn
from phenome.utils import ModelWrapper

# Helper scripts live in Notebooks/examples/sted_fm_utils, next to this notebook.
examples_dir = Path.cwd()
if not (examples_dir / "sted_fm_utils").is_dir():
    examples_dir = examples_dir / "Notebooks" / "examples"
sys.path.insert(0, str(examples_dir.resolve()))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

#### Choose dataset and model

**NAS:** 224×224 crops of STED images of PSD-95 under four experimental conditions: 0MgGlyBic, 48hTTX, Block, and GluGly.

**Optim:** 224×224 crops of STED images of four proteins: CaMKII, PSD-95, actin, and tubulin.

**Zooniverse:** crops of individual synaptic protein clusters. This is the only dataset with masks appropriate for extracting properties.


In [2]:
# Choose one of 3 datasets:
DATASET_NAME = "NAS"
#DATASET_NAME = "Optim"
#DATASET_NAME = "Zooniverse"

# Choose one of 2 models from ("STED-FM", "DinoV2")
MODEL = "STED-FM"
#MODEL = "DinoV2"

#### Download and reshape the dataset

`DATASETS_PATH` is the local folder the archives are saved in. Change it if you want them somewhere else. The helpers in `sted_fm_utils` write `images/`, `masks/` when the dataset has them, and `metadata.csv`.


In [3]:
# Decide where to save datasets
DATASETS_PATH = "stedfm_datasets"
os.makedirs(DATASETS_PATH, exist_ok=True)

DATA_DIR = os.path.join(DATASETS_PATH, f"{DATASET_NAME}_dataset")
os.makedirs(DATA_DIR, exist_ok=True)

In [5]:
from sted_fm_utils import download_dataset, transform_dataset

download_dataset(DATASET_NAME, DATA_DIR)
transform_dataset(DATASET_NAME, DATA_DIR, DATA_DIR)

NAS dataset already exists!
Done!
Transforming NAS dataset...


Saving tiles: 100%|██████████| 1510/1510 [00:01<00:00, 770.35it/s]

Saved 1510 tiles to stedfm_datasets/NAS_dataset/images and stedfm_datasets/NAS_dataset/masks
Metadata written to stedfm_datasets/NAS_dataset/metadata.csv
Done!


### 1. Wrap the STED-FM pretrained model

In [ ]:
from stedfm import get_pretrained_model_v2

class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        model, cfg = get_pretrained_model_v2(
            name="mae-lightning-small",
            weights="MAE_SMALL_STED",
            in_channels=1,
            as_classifier=True,
        )
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        return self.model.forward_features(tensor)

if MODEL == "STED-FM":
    wrapper = STEDFMWrapper(device=device)
elif MODEL == "DinoV2":
    wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

in_channels 1


/data/agonzalez/anaconda3/envs/dinov3/lib/python3.13/site-packages/stedfm/models/lightly_mae.py:126: FutureWarning:

MAEDecoderTIMM is deprecated and will be removed in lightly 1.7.0. Use MaskedVisionTransformerDecoderTIMM instead.



--- mae-lightning-small | (https://s3.valeria.science/flclab-foundation-models/models/mae-small-sted.zip) Downloading from URL and extracting zip ---



100%|██████████| 263M/263M [00:29<00:00, 9.27MB/s] 


Extracting mae-small-sted.zip to /home/agonzalez/.stedfm ...
Loading state_dict from /home/agonzalez/.stedfm/baselines/mae-small_STED/pl_checkpoint-999.pth
--- Loaded model mae-lightning-small with weights MAE_SMALL_STED ---
--- Added linear probe to all frozen blocks ---


### 2. Use custom transforms

The default PhenoMe transforms apply standardisation based on ImageNet values, which isn't appropriate for the STED-FM model.

In [7]:
import torchvision.transforms as T

TARGET_SIZE = 224

if MODEL == "STED-FM":
    # Transforms for STED-FM model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
    ])
elif MODEL == "DinoV2":
    # Transforms for DinoV2 model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])

### 3. Load data (find_files)

We assume CSV-based metadata. The STED datasets were already normalized. If raw data is used, it should be normalized based on percentile values from **entire images**, not by crop.

In [ ]:
import os

images_dir = os.path.join(DATA_DIR, "images")
masks_dir = os.path.join(DATA_DIR, "masks")  # Optional
metadata_dir = os.path.join(DATA_DIR, "metadata.csv")

metadata_df = pd.read_csv(metadata_dir)
metadata_fn = make_dataframe_metadata_fn(metadata_df)
file_df = pheno.find_files(images_dir, mask_dir=masks_dir, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")

Found 1510 files total.
Matched masks: 1510/1510 images (0 missing).
DataFrame: 1510 files, 7 columns.


Found 1510 images


### 4. Process images

In [9]:
# Base path to outputs: checkpoints and plots
OUTPUT_PATH = "output"
os.makedirs(OUTPUT_PATH, exist_ok=True)

CHECKPOINTS_PATH = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

In [10]:
checkpoint_path = os.path.join(CHECKPOINTS_PATH, DATASET_NAME + f"_{MODEL}" + "_checkpoint.hdf5")

if MODEL == "STED-FM":
    pheno.process_images(wrapper, 
                        force_rgb=False,  # STED-FM model expects a single channel
                        checkpoint_path=checkpoint_path,
                        custom_transformations=custom_transforms,
                        )
elif MODEL == "DinoV2":
    pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path=checkpoint_path,
                    custom_transformations=custom_transforms,
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

Processing pipeline:
  1. Read images: native dtype float32, shape 224x224x1, range 0-1
  2. Layout to height x width x channels: channel axis inferred (1 channel)
  3. Channel selection: all channels
  4. Channel handling: split; each channel processed separately
  5. Your transforms: `Compose` replaces default pad, resize, and normalization
  6. Vision model: STEDFMWrapper → embedding
  7. Embedding adjustment: L2-normalize each channel embedding, then concatenate
Custom transforms `Compose` replace PhenoMe's default pad, resize, and normalization stack. Internals are unknown.


Processing batches:   0%|          | 0/48 [00:00<?, ?it/s]

Pipeline ready: 1510 images available (lazy loading from output/checkpoints/NAS_STED-FM_checkpoint.hdf5).
Masks: 1510/1510 images.
Property processing pipeline:
  1. Read images: native dtype float32, shape 224x224x1, range 0-1
  2. Layout to height x width x channels: channel axis inferred (1 channel)
  3. Scale intensities: already 0-1, TypeMaxNorm is identity
  M1. Read masks: native dtype float32, shape 224x224x1, range 0-1
  M2. Layout masks to height x width x channels: channel axis inferred (1 channel)
  M3. Binarize: foreground = pixels > 0.5
  4. Compute properties: preset `complete`


Computing properties:   0%|          | 0/1510 [00:00<?, ?img/s]

Checkpoint: saved 1510 properties (refreshed).


,image_index,image_path,image_name,euler_number,intensity_std_masked,major_axis_length,area,ring_3_mean,ring_1_std,ring_2_std,...,intensity_mean_masked,ring_3_std,intensity_max_masked,ring_2_mean,extent,minor_axis_length,texture_correlation,intensity_min_masked,intensity_min,perimeter
0,0,stedfm_datasets/NAS_dataset/images/img_0.tif,img_0.tif,1,0.066548,37.370056,703,0.256854,0.046945,0.062570,...,0.203868,0.059036,0.367381,0.222315,0.646140,27.311495,0.896020,0.065023,0,129.539108
1,1,stedfm_datasets/NAS_dataset/images/img_1.tif,img_1.tif,12,0.056797,233.933044,1799,0.179111,0.039063,0.054436,...,0.079620,0.045025,0.286102,0.108345,0.053187,169.132462,0.941910,0.000000,0,549.173645
2,2,stedfm_datasets/NAS_dataset/images/img_10.tif,img_10.tif,5,0.041422,219.166946,1109,0.148462,0.029412,0.036005,...,0.077339,0.030700,0.206449,0.100219,0.065722,93.578438,0.940762,0.004877,0,277.320862
3,3,stedfm_datasets/NAS_dataset/images/img_100.tif,img_100.tif,26,0.030745,270.472198,4676,0.092531,0.021946,0.031562,...,0.045683,0.034695,0.145820,0.058229,0.102328,177.391144,0.934763,0.001757,0,1327.040771
4,4,stedfm_datasets/NAS_dataset/images/img_1000.tif,img_1000.tif,20,0.095123,263.544800,3145,0.310530,0.068506,0.093580,...,0.191018,0.078992,0.537190,0.249560,0.068824,233.339233,0.904405,0.008264,0,930.814331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1505,1505,stedfm_datasets/NAS_dataset/images/img_995.tif,img_995.tif,9,0.296152,314.707581,1436,0.675167,0.229342,0.305344,...,0.379785,0.257888,1.000000,0.481035,0.029535,158.460098,0.945246,0.008264,0,423.883301
1506,1506,stedfm_datasets/NAS_dataset/images/img_996.tif,img_996.tif,20,0.120139,280.429047,4060,0.391048,0.088453,0.115079,...,0.246833,0.128880,0.727273,0.289977,0.080915,229.477615,0.914148,0.008264,0,1121.761597
1507,1507,stedfm_datasets/NAS_dataset/images/img_997.tif,img_997.tif,16,0.111334,282.445770,3255,0.332191,0.079178,0.111067,...,0.236163,0.110282,0.702479,0.279870,0.066353,192.056732,0.916370,0.008264,0,884.837646
1508,1508,stedfm_datasets/NAS_dataset/images/img_998.tif,img_998.tif,15,0.110393,288.093079,2194,0.325203,0.078507,0.106912,...,0.205184,0.128527,0.636364,0.242675,0.046201,151.618835,0.906348,0.008264,0,697.298584


### 5. Explore results

In [ ]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

### Generate plots

In [13]:
PLOTS_PATH = os.path.join(OUTPUT_PATH, "plots")
os.makedirs(PLOTS_PATH, exist_ok=True)

if DATASET_NAME == "NAS":
    color_by = "condition"
    fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg", exclude={"condition":"Block"})
    fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_noblock_scatterplot.svg"))
elif DATASET_NAME == "Optim":
    color_by = "label"
elif DATASET_NAME == "Zooniverse":
    color_by = "protein"
    fig=pheno.plot_tsne(color_by="synapse_class", return_fig=True, render_mode="svg")
    fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_noblock_scatterplot.svg"))

fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg")
fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_scatterplot.svg"))
print(f"Plots saved in {PLOTS_PATH}")

Computing t-SNE (1018 samples, 384 features → 2 components)...
Computing t-SNE (1510 samples, 384 features → 2 components)...


Plots saved in output/plots
